# Mốc 8, bước 5: xuất model đã fine-tune sang ONNX / TFLite + đóng gói bộ đánh giá

Không cần GPU. Kết quả lưu vào `MyDrive/doc-scanner-ocr/export/`:
- `vi_rec.onnx`, `vi_rec_float32.tflite`: model đọc chữ tiếng Việt (từ `output/best_accuracy`)
- `vi_dict.txt`: từ điển
- `eval_set.zip`: 1.500 dòng đánh giá (font NotoSerif) để chấm trên Mac bằng `tools/ocr_eval.py`

In [ ]:
# 1. Drive + dữ liệu
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/doc-scanner-ocr"
!mkdir -p $OUT/export
!cd /content && rm -rf data && unzip -q $OUT/data.zip && wc -l data/eval.txt

In [ ]:
# 2. Cài PaddlePaddle (bản CPU là đủ để xuất model) + PaddleOCR v3.7.0 + công cụ convert
!pip -q install paddlepaddle==3.2.0
![ -d /content/PaddleOCR ] || git clone -q --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
!pip -q install -r /content/PaddleOCR/requirements.txt
# Cài từng nhóm riêng, hiện dòng cuối của mỗi lệnh: một gói lỗi không làm hỏng các gói khác
!pip install onnx onnxruntime 2>&1 | tail -2
!pip install onnx2tf onnxsim onnx_graphsurgeon sng4onnx tf_keras ai-edge-litert 2>&1 | tail -2
# paddle2onnx chỉ hỗ trợ tới Python 3.12, Colab đang dùng 3.13 -> tạo môi trường Python 3.12 riêng (uv) chỉ để chạy nó
!pip -q install uv && uv venv -q --clear -p 3.12 /content/py312 && uv pip install -q --python /content/py312/bin/python paddlepaddle==3.2.0 paddle2onnx==2.1.0
!/content/py312/bin/paddle2onnx --version
!python -c "import onnxruntime, paddle; print('onnxruntime', onnxruntime.__version__, '| paddle', paddle.__version__)"

In [ ]:
# 3. Xuất trọng số best_accuracy thành model suy luận (chỉ giữ nhánh CTC, bỏ nhánh phụ NRTR chỉ dùng lúc train)
%cd /content/PaddleOCR
!python tools/export_model.py -c configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml -o \
  Global.pretrained_model=$OUT/output/best_accuracy Global.character_dict_path=/content/data/vi_dict.txt \
  Global.save_inference_dir=/content/vi_rec_infer 2>&1 | tail -3
!ls -la /content/vi_rec_infer

In [ ]:
# 4. Paddle -> ONNX bằng paddle2onnx trong môi trường Python 3.12 (tự nhận file inference.json hoặc inference.pdmodel)
import os
d = "/content/vi_rec_infer"
mf = "inference.json" if os.path.exists(f"{d}/inference.json") else "inference.pdmodel"
!/content/py312/bin/paddle2onnx --model_dir {d} --model_filename {mf} --params_filename inference.pdiparams --save_file /content/vi_rec.onnx --opset_version 14 2>&1 | tail -3
import onnxruntime as ort
s = ort.InferenceSession("/content/vi_rec.onnx", providers=["CPUExecutionProvider"])
print("vao:", [(i.name, i.shape) for i in s.get_inputs()], "| ra:", [(o.name, o.shape) for o in s.get_outputs()])

In [ ]:
# 5. ONNX -> TFLite (cố định ảnh vào 1x3x48x320, giống bước 2)
inp = s.get_inputs()[0].name
!onnx2tf -i /content/vi_rec.onnx -o /content/vi_tflite -ois {inp}:1,3,48,320 -n 2>&1 | tail -2
!ls -la /content/vi_tflite

In [ ]:
# 6. Kiểm tra nhanh: ONNX và TFLite đọc 10 dòng eval giống nhau và giống đáp án
import numpy as np, cv2, glob
from ai_edge_litert.interpreter import Interpreter
chars = ["<blank>"] + [l.rstrip("\n") for l in open("/content/data/vi_dict.txt", encoding="utf-8") if l.rstrip("\n")] + [" "]
def prep(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB); h, w = rgb.shape[:2]; nw = min(320, int(np.ceil(48 * w / h)))
    im = (cv2.resize(rgb, (nw, 48)).astype(np.float32) / 255.0 - 0.5) / 0.5
    out = np.zeros((48, 320, 3), np.float32); out[:, :nw] = im; return out.transpose(2, 0, 1)[None]
def ctc(p):
    res, prev = [], -1
    for k in p.argmax(-1):
        if k != prev and k != 0: res.append(chars[k])
        prev = k
    return "".join(res)
tfl = sorted(glob.glob("/content/vi_tflite/*float32.tflite"))[0]
it = Interpreter(model_path=tfl); it.allocate_tensors(); di, do = it.get_input_details()[0], it.get_output_details()[0]
for line in open("/content/data/eval.txt", encoding="utf-8").readlines()[:10]:
    p, gt = line.rstrip("\n").split("\t")
    x = prep(cv2.imread("/content/data/" + p))
    a = ctc(s.run(None, {inp: x})[0][0])
    it.set_tensor(di["index"], x.transpose(0, 2, 3, 1).copy()); it.invoke(); b = ctc(it.get_tensor(do["index"])[0])
    print(f"{gt!r:30} ONNX {a!r:30} TFLite {b!r}")

In [ ]:
# 7. Lưu lên Drive: model + từ điển + bộ đánh giá
!cp /content/vi_rec.onnx $OUT/export/ && cp {tfl} $OUT/export/vi_rec_float32.tflite && cp /content/data/vi_dict.txt $OUT/export/
!cd /content/data && zip -q -r $OUT/export/eval_set.zip eval eval.txt vi_dict.txt
!ls -la $OUT/export